# Milestone 1 — Setup, sample footage, pretrained YOLO

**Goal:** get a broadcast clip, run an off-the-shelf YOLO detector on it, and *measure what it gets wrong*.
Those failures are the motivation for every later milestone.

**Works on:** local machine (NVIDIA GPU / Apple Silicon / CPU) or Google Colab / Kaggle (Runtime → Change runtime type → **T4 GPU**).

### Deep-learning crash course (only what you need today)
| Term | Plain meaning | Analogy from your tabular work |
|---|---|---|
| **Neural network** | A big function with millions of tunable numbers (*weights*) that maps pixels → predictions | Like LightGBM's trees, but the "features" are learned from raw pixels instead of engineered by you |
| **CNN / backbone** | Layers that slide small filters over the image to detect edges → shapes → objects | Automatic feature engineering |
| **Object detector (YOLO)** | Outputs, for each object: a **box** (x1,y1,x2,y2), a **class**, and a **confidence** in one pass ("You Only Look Once") | A classifier + a regressor for box coordinates, run on a grid over the image |
| **Pretrained weights / COCO** | Weights already trained on COCO (330k images, 80 everyday classes incl. `person`, `sports ball`) | Using a model someone else fitted, then just calling `.predict()` |
| **Inference** | Running the trained model on new data — no learning happens | `model.predict(X_test)` |
| **GPU / CUDA** | Graphics card doing the matrix maths in parallel; CUDA is NVIDIA's API for it | 10–50× faster than CPU for this |

## 1. Environment

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    !pip install -q ultralytics supervision kaggle
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE)

In [ ]:
from pathlib import Path
import time
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, ultralytics, supervision as sv
from ultralytics import YOLO

# Repo root: notebooks/ lives one level below it locally; on Colab/Kaggle use a working folder
ROOT = Path("/content/football-tracking") if IN_COLAB else (
       Path("/kaggle/working/football-tracking") if IN_KAGGLE else Path.cwd().parent)
for sub in ["data/raw", "data/frames", "models", "outputs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch {torch.__version__} | ultralytics {ultralytics.__version__} | supervision {sv.__version__}")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("ROOT:", ROOT)

## 2. Get sample footage

| Source | What you get | Access | Use it for |
|---|---|---|---|
| **Kaggle — DFL Bundesliga Data Shootout** | ~30 s broadcast clips (1080p, 25 fps) from Bundesliga games | Free, accept competition rules | **Main demo clips (start here)** |
| **Roboflow Universe — "football-players-detection"** | ~300 labelled frames: ball / goalkeeper / player / referee | Free account, API key | Fine-tuning in **Milestone 2** |
| **Roboflow Universe — "football-field-detection"** | Labelled pitch keypoints | Free account | Homography in **Milestone 6** |
| **SoccerNet** (tracking, game-state, calibration tasks) | Large research datasets with tracking IDs & calibration | Free, but full videos need an NDA form | Evaluation / stretch goals |
| Your own clip | YouTube highlights, etc. | Check licence before putting it in the README GIF | Final demo |

👉 **Pick a "tactical-ish" wide-angle clip** for development: main broadcast camera, few close-ups/replays. Close-ups and replay cuts break tracking and homography later.

### Option A — Kaggle API
1. Kaggle → Settings → **Create New Token** → downloads `kaggle.json`
2. Accept rules: https://www.kaggle.com/competitions/dfl-bundesliga-data-shootout/rules
3. Run the cell below (on Colab it asks you to upload `kaggle.json`).

In [ ]:
CLIP_NAME = "08fd33_4.mp4"          # any file from the competition's clips/ folder
VIDEO = ROOT / "data/raw" / CLIP_NAME

if not VIDEO.exists():
    if IN_COLAB:
        from google.colab import files
        print("Upload kaggle.json"); files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c dfl-bundesliga-data-shootout -f clips/{CLIP_NAME} -p {ROOT/'data/raw'}
    zipped = VIDEO.with_suffix(VIDEO.suffix + ".zip")
    if zipped.exists():
        !unzip -o -q {zipped} -d {ROOT/'data/raw'} && rm {zipped}
print(VIDEO, "exists:", VIDEO.exists())

### Option B — your own clip
Drop any `.mp4` into `data/raw/` (on Colab: `files.upload()`), then set `VIDEO = ROOT/"data/raw"/"yourclip.mp4"`.
Keep it **≤ 30 s** while developing — every experiment reruns over every frame.

## 3. Inspect the video

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
info = dict(width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            fps=cap.get(cv2.CAP_PROP_FPS), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()
info["duration_s"] = round(info["frames"] / info["fps"], 1)
info

In [ ]:
def read_frame(path, idx):
    cap = cv2.VideoCapture(str(path)); cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read(); cap.release()
    assert ok, f"could not read frame {idx}"
    return f  # NOTE: OpenCV gives BGR, matplotlib expects RGB

def show(img_bgr, title="", size=(14, 8)):
    plt.figure(figsize=size); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis("off"); plt.show()

idxs = np.linspace(0, info["frames"] - 1, 4, dtype=int)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, i in zip(axes.ravel(), idxs):
    ax.imshow(cv2.cvtColor(read_frame(VIDEO, i), cv2.COLOR_BGR2RGB)); ax.set_title(f"frame {i}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Save every 25th frame (1 per second) — handy for eyeballing now and for labelling later
cap = cv2.VideoCapture(str(VIDEO)); i = saved = 0
while True:
    ok, f = cap.read()
    if not ok: break
    if i % 25 == 0:
        cv2.imwrite(str(ROOT / "data/frames" / f"{VIDEO.stem}_{i:06d}.jpg"), f); saved += 1
    i += 1
cap.release(); print("saved", saved, "frames to data/frames/")

## 4. Pretrained YOLO on one frame
Model sizes: `n` (nano) < `s` < `m` < `l` < `x`. Bigger = more accurate, slower. Weights download automatically the first time.

We keep only two COCO classes: **0 = person**, **32 = sports ball**.

In [ ]:
model = YOLO("yolo11m.pt")
print({k: v for k, v in model.names.items() if k in (0, 32)})

frame = read_frame(VIDEO, idxs[1])
res = model.predict(frame, classes=[0, 32], conf=0.25, imgsz=1280, device=DEVICE, verbose=False)[0]
det = sv.Detections.from_ultralytics(res)   # supervision's standard container: .xyxy, .confidence, .class_id
print(len(det), "detections")

In [ ]:
# Detections as a DataFrame — your comfort zone
df = pd.DataFrame(det.xyxy, columns=["x1", "y1", "x2", "y2"])
df["cls"] = det.data["class_name"]; df["conf"] = det.confidence
df["w"], df["h"] = df.x2 - df.x1, df.y2 - df.y1
df.sort_values("conf").head(10)

In [ ]:
box_ann, lbl_ann = sv.BoxAnnotator(thickness=2), sv.LabelAnnotator(text_scale=0.5)
def annotate(img, d):
    labels = [f"{n} {c:.2f}" for n, c in zip(d.data["class_name"], d.confidence)]
    return lbl_ann.annotate(box_ann.annotate(img.copy(), d), d, labels=labels)

show(annotate(frame, det), "yolo11m, COCO, imgsz=1280")

In [ ]:
# Box size distribution: why the ball is hard
display(df.groupby("cls")[["w", "h", "conf"]].describe().T.round(1))

## 5. Experiments (do these — they teach you the knobs)
### 5a. Input resolution vs. ball detection
YOLO resizes the frame to `imgsz` before predicting. At 640 a 1920-px frame is shrunk 3× → a 15-px ball becomes 5 px and vanishes.

In [ ]:
sample_idxs = np.linspace(0, info["frames"] - 1, 40, dtype=int)
sample = [read_frame(VIDEO, i) for i in sample_idxs]

rows = []
for imgsz in [640, 1280]:
    t0 = time.time()
    results = model.predict(sample, classes=[0, 32], conf=0.15, imgsz=imgsz, device=DEVICE, verbose=False)
    dt = (time.time() - t0) / len(sample)
    ds = [sv.Detections.from_ultralytics(r) for r in results]
    rows.append(dict(imgsz=imgsz, ms_per_frame=round(dt * 1000),
                     ball_found_pct=100 * np.mean([(d.class_id == 32).any() for d in ds]),
                     avg_persons=np.mean([(d.class_id == 0).sum() for d in ds])))
pd.DataFrame(rows)

### 5b. Model size vs. speed

In [ ]:
rows = []
for w in ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]:
    m = YOLO(w)
    m.predict(sample[:2], imgsz=1280, device=DEVICE, verbose=False)  # warm-up (first call is always slow)
    t0 = time.time()
    results = m.predict(sample, classes=[0, 32], conf=0.15, imgsz=1280, device=DEVICE, verbose=False)
    dt = (time.time() - t0) / len(sample)
    ds = [sv.Detections.from_ultralytics(r) for r in results]
    rows.append(dict(weights=w, ms_per_frame=round(dt * 1000), fps=round(1 / dt, 1),
                     ball_found_pct=100 * np.mean([(d.class_id == 32).any() for d in ds]),
                     avg_persons=np.mean([(d.class_id == 0).sum() for d in ds])))
pd.DataFrame(rows)

### 5c. Confidence threshold
Lower `conf` → more true balls found **and** more false balls (heads, socks, penalty spot, white boots).
Try `conf` ∈ {0.05, 0.15, 0.3} on one frame and look at the pictures, not just the counts.

In [ ]:
for c in [0.05, 0.15, 0.30]:
    r = model.predict(frame, classes=[32], conf=c, imgsz=1280, device=DEVICE, verbose=False)[0]
    d = sv.Detections.from_ultralytics(r)
    print(f"conf={c}: {len(d)} ball candidates, confidences={np.round(d.confidence, 2)}")

## 6. Full clip → annotated video

In [ ]:
OUT = ROOT / "outputs" / f"m1_{VIDEO.stem}_yolo11m.mp4"
vinfo = sv.VideoInfo.from_video_path(str(VIDEO))
stats, t0 = [], time.time()
with sv.VideoSink(str(OUT), vinfo) as sink:
    for i, f in enumerate(sv.get_video_frames_generator(str(VIDEO))):
        r = model.predict(f, classes=[0, 32], conf=0.2, imgsz=1280, device=DEVICE, verbose=False)[0]
        d = sv.Detections.from_ultralytics(r)
        stats.append(dict(frame=i, persons=int((d.class_id == 0).sum()), balls=int((d.class_id == 32).sum())))
        sink.write_frame(annotate(f, d))
print(f"{len(stats)} frames in {time.time() - t0:.0f}s -> {OUT}")

stats = pd.DataFrame(stats)
print(f"Ball visible in {100 * (stats.balls > 0).mean():.1f}% of frames | "
      f"frames with >1 ball: {(stats.balls > 1).sum()} | persons/frame: {stats.persons.mean():.1f}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(stats.frame, stats.persons); ax[0].set_ylabel("# persons")
ax[1].step(stats.frame, stats.balls, where="post", color="tab:orange"); ax[1].set_ylabel("# balls"); ax[1].set_xlabel("frame")
plt.suptitle("Detections over time — gaps in the ball row = what Milestone 5 must fix"); plt.show()

In [ ]:
# Colab: download the video to watch it
if IN_COLAB:
    from google.colab import files; files.download(str(OUT))

## 7. Same thing with the clean module
The repo's `src/football_tracking/` contains the tidy version of this notebook.
```bash
python scripts/run_detection.py --video data/raw/08fd33_4.mp4 --out outputs/m1_detect.mp4
```

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
try:
    from football_tracking.detection import Detector, Annotator, summarize
    d = Detector("yolo11m.pt"); print(summarize([d.detect(f) for f in sample[:10]]))
except ModuleNotFoundError:
    print("src/ not found here — on Colab, clone your repo into", ROOT, "or upload the src folder.")

## 8. Failure log — write your own observations here
Watch the output video and note concrete examples (frame numbers!). Expected ones:

- **`person` ≠ player.** Referees, linesmen, coaches, ball boys, fans in the stand are all `person`. → *Milestone 2 (fine-tune with separate classes).*
- **Ball missing** in many frames (small, motion-blurred, occluded by legs). → *M2 + M5 interpolation.*
- **False balls** — heads, white boots, logos, penalty spot. → *M2 + M5 "keep one ball" logic.*
- **No identity.** Box #3 in frame 10 has no link to any box in frame 11. → *M3 ByteTrack.*
- **Overlapping players** merged into one box or one lost. → *ID switches in M3.*
- **Camera pans/zooms** — pixel positions are not pitch positions. → *M6 homography.*
- **Close-ups / replays** — completely different scale. → filter or cut these out of your demo clip.

My notes:
- frame ___ : ...

## ✅ Before Milestone 2, make sure you can answer
1. What do a detection's three outputs (box, class, confidence) mean, and what do `conf` and `imgsz` change?
2. Why does `imgsz=1280` find the ball more often, and what does it cost?
3. Why is a COCO model not enough for this project (two reasons)?
4. Difference between *training* and *inference*; what "pretrained weights" are.
5. Why OpenCV images look blue-tinted in matplotlib (BGR vs RGB).
6. Your numbers: FPS for n/s/m on your hardware, and ball-visible % for your clip.